# Handling Missing Data in Spark

Working with nulls using na.drop, na.fill, and na.replace

If yesterday’s joins felt like wiring two clean tables together, today is about reality: data is rarely clean. You’ll see empty strings sneaking into numeric columns, NULL values sprinkled across key fields, and the occasional “NA” or “-” pretending to be a number. As a data engineer, your job isn’t to be upset by it—you make the data trustworthy.

Spark gives you three sturdy tools on every DataFrame via the .na helper: drop, fill, and replace. We’ll use them in a way that feels natural and safe, and we’ll talk about when not to use them too.

We’ll stick with a slightly messier version of our sales data:

Notice the trouble: revenue has blanks, "NA", and missing; store is missing once. This is realistic.

###  Load — and teach Spark what “null” looks like

When reading CSVs, Spark can treat certain strings as nulls right away. That tiny decision saves a lot of “cleaning later.”

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim

spark = SparkSession.builder.appName("day11-missing").getOrCreate()

df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)       # good for learning; in prod, define schemas explicitly
         .option("nullValue", "NA")         # strings to interpret as null
         .option("emptyValue", None)        # empty fields → null (helpful for blanks)
         .csv("/Volumes/thedataengineering_00/data/sampledata/salesdirty/")
)

df.show(truncate=False)
df.printSchema()

If your source sometimes uses whitespace as a “blank,” emptyValue won’t catch a cell that contains a space. A small normalization step helps:

In [0]:
# Turn pure-whitespace strings into nulls for *all* string columns
from pyspark.sql.functions import when, length

for c, t in df.dtypes:
    if t == "string":
        df = df.withColumn(c, when(length(trim(col(c))) == 0, None).otherwise(col(c)))

display(df)


This gives Spark a fighting chance to separate “real data” from “no data.”

### See the missingness — don’t guess
Before you fix anything, measure it. You need to know how bad the problem is and where it lives.

In [0]:
for c in df.columns:
    null_count = df.filter(df[c].isNull()).count()
    print(f"{c}: {null_count}")

This quick view tells you, column by column, how many nulls you’re dealing with. If revenue has dozens of nulls and store has one, your strategy might differ per column.

### When rows are too broken — na.drop
There are times when a row simply isn’t usable. Maybe both store and revenue are missing. You can drop rows with missing values selectively.

In [0]:
# Drop rows that are missing *any* of the listed columns
df_drop_any = df.na.drop(subset=["store", "revenue"])
# Keep only rows that have at least N non-null fields overall (thresh)
df_keep_min3 = df.na.drop(thresh=3)  # keep rows with 3+ non-null values

Two levers matter here:

subset – focus the rule on critical columns (e.g., rows without revenue are useless for revenue KPIs).
how / thresh – how="any" drops if any listed column is null; thresh keeps rows with at least N non-nulls.

Be intentional. Dropping rows is irreversible for that stage of the pipeline and can bias metrics if used casually.

### When imputation makes sense — na.fill
Often you don’t want to drop; you want to fill. For numeric KPIs, a 0 may be acceptable; for text, “Unknown” is safer than pretending it never happened.

In [0]:
# Fill different columns with different defaults
df_filled = df.na.fill({
    "revenue": 0,          # numeric default
    "store":   "Unknown"   # string default
})

df_filled.show(truncate=False)

A few nuances worth knowing:

fill works per data type. If you pass a scalar (like 0), Spark applies it to all numeric columns; a string default applies to all string columns. Using a dict lets you be precise per column.

fill treats NaN as missing for floating types. If your revenue is double/float and contains NaN, fill(0) covers both null and NaN.

Choose defaults with care. Filling missing revenue with 0 may be okay for rollups (it won’t inflate totals) but could be misleading if a null meant “not reported yet” versus “truly zero sales.” If you want to impute with a statistic (mean/median), compute it first:

In [0]:
from pyspark.sql.functions import avg
 
avg_rev = df.select(avg(col("revenue"))).first()[0]
print(avg_rev)
df_imputed = df.na.fill({"revenue": str(avg_rev)})

df_imputed.show()

### When the problem isn’t “null” but a bad value — na.replace

Sometimes the data isn’t missing — it’s wrong. Common culprits: “N/A”, “-”, “unknown”, or the classic "0000-00-00" as a fake date. na.replace lets you swap values (not just nulls).

In [0]:
# Replace specific sentinels with null or better labels
df_filled.show()

df_clean = (
    df_filled.na.replace({'NULL': "0"}, subset=["revenue"])  # to nulls
      .na.replace({"Unknown": 'Store Standard'}, subset=["store"])           # standardize Unknowns as nulls
)

df_clean.show(truncate=False)

You can also replace legitimate values — e.g., renaming a region:

In [0]:
df.show()
df_reg = df.na.replace({"North": "N", "South": "S"}, subset=["region"])
df_reg.show()

Think of replace as “value mapping,” while fill is “fill the missing holes,” and drop is “discard beyond repair.”

Nulls and calculations — what really happens?
Understanding how aggregations treat nulls will save you from head-scratching:

- sum, avg, max, min ignore nulls (they operate on non-null values).
- count("*") counts all rows; count("col") counts non-null values in col.

In [0]:
from pyspark.sql.functions import sum as Fsum, avg as Favg, count

(
    df.select(
        Fsum("revenue").alias("sum_rev"),
        Favg("revenue").alias("avg_rev"),
        count("*").alias("rows_total"),
        count("revenue").alias("rows_with_revenue")
    )
).show()

If you see a big gap between rows_total and rows_with_revenue, you know nulls are common and your averages reflect only the reported subset.

### After joins — cleaning at the seams
Joins routinely produce nulls on the “missing side.” That’s not an error; it’s information. You can decide what to do next:

In [0]:
# Suppose df_left joined df_right on key; missing matches produce nulls on one side.
# Example: fill missing dimension attributes with safe labels
df_joined_safe = df.na.fill({
    "store": "Unknown",
    "region": "Unassigned"
})

Prefer leaving nulls as null until after you’ve done diagnostics; they help you find data coverage issues. Fill only when you’re clear on the business meaning.

### A small, realistic pipeline — reading, normalizing, filling, validating
Let’s put it together as a tidy flow you can reuse:

In [0]:
from pyspark.sql.functions import when, length, trim, col, avg

df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .option("nullValue", "NA")
         .option("emptyValue", None)
         .csv("/Volumes/thedataengineering_00/data/sampledata/salesdirty/")
)

# Normalize whitespace-only strings → null
for c, t in df.dtypes:
    if t == "string":
        df = df.withColumn(c, when(length(trim(col(c))) == 0, None).otherwise(col(c)))

df = df.withColumn("revenue", col("revenue").cast("float"))

# Impute revenue: fill with 0 (or choose mean if business prefers)
mean_rev = df.select(avg(col("revenue"))).first()[0]
df = df.na.fill({"revenue": mean_rev})   # or {"revenue": float(mean_rev) if mean_rev else 0.0}
print("dropped revenue and store with null values")
df.show()

# Drop Null store and revenue
df = df.na.drop(subset=["store", "revenue"])
print("dropped revenue and store with null values")
df.show()

# Replace specific sentinels with null or better labels
df = df.na.replace({'North': "N", "South": "S", "East": "E", "West": "W"}, subset=["region"])  
print("replaced region with N, S, E, W")
df.show()


# Describe Show
df.describe().show()

You read intentionally, normalized strings, dropped only what was irredeemable, filled in a documented way, and left yourself a final check. That’s professional-grade data hygiene.

When not to fill or drop
It’s tempting to “just fill 0” everywhere and move on. Resist that urge. Ask:

- Does a missing revenue mean zero sales, or report not received?
- If I drop rows, do I bias the dashboard toward “cleaner” regions or stores?
- Will my model learn the wrong signal if I flatten missingness into an ordinary value?

Sometimes the right answer is to keep the null, carry it forward, and let the downstream consumer decide how to treat it. Your job is to make the missingness visible and intentional.

Wrap-up
Missing data isn’t a nuisance; it’s a signal. With Spark’s .na tools:

- drop removes rows that fail your minimum data contract.
- fill supplies safe defaults or imputations—carefully, per column and data type.
- replace standardizes bad sentinels and remaps values cleanly.

Pair those with good reading options (nullValue, emptyValue) and a habit of measuring nulls before and after you clean, and you’ll ship pipelines that analysts can trust.

Tomorrow, we’ll climb into dates and timestamps — because time, like missingness, has sharp edges.